In [2]:
# ============================================================
# ENVIRONMENT & CONFIGURATION
# ============================================================

from pathlib import Path

import numpy as np
import polars as pl

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent.resolve()

SPI_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "spi"
)

SPI3_FILE = SPI_DIR / "spi3.parquet"
SPI6_FILE = SPI_DIR / "spi6.parquet"
SPI12_FILE = SPI_DIR / "spi12.parquet"

# ------------------------------------------------------------
# Validation constants
# ------------------------------------------------------------

SPI_FILES = {
    "SPI-3": SPI3_FILE,
    "SPI-6": SPI6_FILE,
    "SPI-12": SPI12_FILE,
}

EXPECTED_H3_CELLS = 42_316
EXPECTED_TIMESTAMPS = 276
EXPECTED_ROWS = (
    EXPECTED_H3_CELLS
    * EXPECTED_TIMESTAMPS
)

print("=" * 60)
print("SPI VALIDATION")
print("=" * 60)

for name, file in SPI_FILES.items():

    print(f"{name:<8}: {file}")

    assert file.exists(), (
        f"Missing file: {file}"
    )

print("=" * 60)
print(f"Expected H3 cells : {EXPECTED_H3_CELLS:,}")
print(f"Expected months   : {EXPECTED_TIMESTAMPS}")
print(f"Expected rows     : {EXPECTED_ROWS:,}")
print("=" * 60)

SPI VALIDATION
SPI-3   : C:\Projects\ews\zoryn\data\processed\spi\spi3.parquet
SPI-6   : C:\Projects\ews\zoryn\data\processed\spi\spi6.parquet
SPI-12  : C:\Projects\ews\zoryn\data\processed\spi\spi12.parquet
Expected H3 cells : 42,316
Expected months   : 276
Expected rows     : 11,679,216


In [3]:
# ============================================================
# LOAD SPI DATASETS
# ============================================================

spi = {}

print("=" * 60)
print("LOAD SPI DATASETS")
print("=" * 60)

for name, file in SPI_FILES.items():

    df = (
        pl.scan_parquet(file)
        .sort(
            [
                "h3_id",
                "timestamp",
            ]
        )
        .collect()
    )

    spi[name] = df

    summary = (
        df.select(
            [
                pl.len().alias("rows"),
                pl.col("h3_id").n_unique().alias("h3_cells"),
                pl.col("timestamp").n_unique().alias("timestamps"),
                pl.col("timestamp").min().alias("first_timestamp"),
                pl.col("timestamp").max().alias("last_timestamp"),
            ]
        )
    )

    print(name)
    print(summary)
    print()

    assert summary["rows"][0] == EXPECTED_ROWS
    assert summary["h3_cells"][0] == EXPECTED_H3_CELLS
    assert summary["timestamps"][0] == EXPECTED_TIMESTAMPS

print("=" * 60)
print("SPI DATASETS LOADED")
print("=" * 60)

LOAD SPI DATASETS
SPI-3
shape: (1, 5)
┌──────────┬──────────┬────────────┬─────────────────┬────────────────┐
│ rows     ┆ h3_cells ┆ timestamps ┆ first_timestamp ┆ last_timestamp │
│ ---      ┆ ---      ┆ ---        ┆ ---             ┆ ---            │
│ u32      ┆ u32      ┆ u32        ┆ str             ┆ str            │
╞══════════╪══════════╪════════════╪═════════════════╪════════════════╡
│ 11679216 ┆ 42316    ┆ 276        ┆ 2002-01         ┆ 2024-12        │
└──────────┴──────────┴────────────┴─────────────────┴────────────────┘

SPI-6
shape: (1, 5)
┌──────────┬──────────┬────────────┬─────────────────┬────────────────┐
│ rows     ┆ h3_cells ┆ timestamps ┆ first_timestamp ┆ last_timestamp │
│ ---      ┆ ---      ┆ ---        ┆ ---             ┆ ---            │
│ u32      ┆ u32      ┆ u32        ┆ str             ┆ str            │
╞══════════╪══════════╪════════════╪═════════════════╪════════════════╡
│ 11679216 ┆ 42316    ┆ 276        ┆ 2002-01         ┆ 2024-12        │
└────

In [4]:
# ============================================================
# SPI QUALITY ASSURANCE
# ============================================================

print("=" * 60)
print("SPI QUALITY ASSURANCE")
print("=" * 60)

for name, df in spi.items():

    column = name.lower().replace("-", "")

    summary = (
        df.select(
            [
                pl.col(column).is_null().sum().alias("nulls"),
                pl.col(column).is_nan().sum().alias("nans"),
                pl.col(column).is_infinite().sum().alias("infinities"),
                pl.col(column).min().alias("min"),
                pl.col(column).max().alias("max"),
            ]
        )
    )

    print(name)
    print(summary)
    print()

    assert summary["infinities"][0] == 0

print("=" * 60)
print("SPI QA PASSED")
print("=" * 60)

SPI QUALITY ASSURANCE
SPI-3
shape: (1, 5)
┌───────┬───────┬────────────┬───────────┬─────────┐
│ nulls ┆ nans  ┆ infinities ┆ min       ┆ max     │
│ ---   ┆ ---   ┆ ---        ┆ ---       ┆ ---     │
│ u32   ┆ u32   ┆ u32        ┆ f64       ┆ f64     │
╞═══════╪═══════╪════════════╪═══════════╪═════════╡
│ 0     ┆ 84632 ┆ 0          ┆ -2.976072 ┆ 3.93311 │
└───────┴───────┴────────────┴───────────┴─────────┘

SPI-6
shape: (1, 5)
┌───────┬────────┬────────────┬───────────┬──────────┐
│ nulls ┆ nans   ┆ infinities ┆ min       ┆ max      │
│ ---   ┆ ---    ┆ ---        ┆ ---       ┆ ---      │
│ u32   ┆ u32    ┆ u32        ┆ f64       ┆ f64      │
╞═══════╪════════╪════════════╪═══════════╪══════════╡
│ 0     ┆ 211580 ┆ 0          ┆ -2.184995 ┆ 3.154816 │
└───────┴────────┴────────────┴───────────┴──────────┘

SPI-12
shape: (1, 5)
┌───────┬────────┬────────────┬───────────┬──────────┐
│ nulls ┆ nans   ┆ infinities ┆ min       ┆ max      │
│ ---   ┆ ---    ┆ ---        ┆ ---       ┆ ---  

In [5]:
# ============================================================
# SPI DISTRIBUTION STATISTICS
# ============================================================

print("=" * 60)
print("SPI DISTRIBUTION STATISTICS")
print("=" * 60)

for name, df in spi.items():

    column = name.lower().replace("-", "")

    values = (
        df.select(pl.col(column))
        .drop_nans()
    )

    summary = (
        values.select(
            [
                pl.len().alias("count"),
                pl.mean(column).alias("mean"),
                pl.std(column).alias("std"),
                pl.median(column).alias("median"),
                pl.min(column).alias("min"),
                pl.max(column).alias("max"),
                pl.quantile(column, 0.01).alias("p01"),
                pl.quantile(column, 0.05).alias("p05"),
                pl.quantile(column, 0.95).alias("p95"),
                pl.quantile(column, 0.99).alias("p99"),
            ]
        )
    )

    print(name)
    print(summary)
    print()

print("=" * 60)
print("STATISTICS COMPLETE")
print("=" * 60)

SPI DISTRIBUTION STATISTICS
SPI-3
shape: (1, 10)
┌──────────┬───────────┬─────────┬──────────┬───┬───────────┬───────────┬──────────┬──────────┐
│ count    ┆ mean      ┆ std     ┆ median   ┆ … ┆ p01       ┆ p05       ┆ p95      ┆ p99      │
│ ---      ┆ ---       ┆ ---     ┆ ---      ┆   ┆ ---       ┆ ---       ┆ ---      ┆ ---      │
│ u32      ┆ f64       ┆ f64     ┆ f64      ┆   ┆ f64       ┆ f64       ┆ f64      ┆ f64      │
╞══════════╪═══════════╪═════════╪══════════╪═══╪═══════════╪═══════════╪══════════╪══════════╡
│ 11594584 ┆ -0.001523 ┆ 0.99855 ┆ -0.13071 ┆ … ┆ -1.842365 ┆ -1.466861 ┆ 1.776341 ┆ 2.589788 │
└──────────┴───────────┴─────────┴──────────┴───┴───────────┴───────────┴──────────┴──────────┘

SPI-6
shape: (1, 10)
┌──────────┬───────────┬──────────┬───────────┬───┬───────────┬───────────┬──────────┬────────┐
│ count    ┆ mean      ┆ std      ┆ median    ┆ … ┆ p01       ┆ p05       ┆ p95      ┆ p99    │
│ ---      ┆ ---       ┆ ---      ┆ ---       ┆   ┆ ---       ┆ -

In [6]:
# ============================================================
# SAMPLE H3 TIME SERIES
# ============================================================

SAMPLE_CELLS = 5

print("=" * 80)
print("SAMPLE SPI TIME SERIES")
print("=" * 80)

sample_ids = (
    spi["SPI-3"]
    .select("h3_id")
    .unique()
    .sort("h3_id")
    .head(SAMPLE_CELLS)
    .to_series()
)

for h3_id in sample_ids:

    print(f"H3: {h3_id}")
    print("-" * 80)

    sample = (
        spi["SPI-3"]
        .filter(pl.col("h3_id") == h3_id)
        .join(
            spi["SPI-6"],
            on=["h3_id", "timestamp"],
        )
        .join(
            spi["SPI-12"],
            on=["h3_id", "timestamp"],
        )
        .select(
            [
                "timestamp",
                "spi3",
                "spi6",
                "spi12",
            ]
        )
    )

    print(sample.head(15))
    print()

SAMPLE SPI TIME SERIES
H3: 8a7a6325b20ffff
--------------------------------------------------------------------------------
shape: (15, 4)
┌───────────┬───────────┬───────────┬──────────┐
│ timestamp ┆ spi3      ┆ spi6      ┆ spi12    │
│ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪══════════╡
│ 2002-01   ┆ NaN       ┆ NaN       ┆ NaN      │
│ 2002-02   ┆ NaN       ┆ NaN       ┆ NaN      │
│ 2002-03   ┆ -0.004294 ┆ NaN       ┆ NaN      │
│ 2002-04   ┆ -0.117566 ┆ NaN       ┆ NaN      │
│ 2002-05   ┆ 0.205188  ┆ NaN       ┆ NaN      │
│ …         ┆ …         ┆ …         ┆ …        │
│ 2002-11   ┆ -0.139328 ┆ -0.339722 ┆ NaN      │
│ 2002-12   ┆ 1.06805   ┆ 1.021294  ┆ 0.58552  │
│ 2003-01   ┆ 0.980998  ┆ 0.823182  ┆ 0.660308 │
│ 2003-02   ┆ 1.191853  ┆ 0.701933  ┆ 0.611908 │
│ 2003-03   ┆ -0.999159 ┆ 0.390088  ┆ 0.395842 │
└───────────┴───────────┴───────────┴──────────┘

H3: 8a7a6325b22ffff
-------

In [7]:
# ============================================================
# COMPARE RAINFALL HISTORIES
# ============================================================

from pathlib import Path

OBS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

observations = (
    pl.scan_parquet(OBS_FILE)
    .filter(pl.col("variable") == "precipitation_amount")
    .select(
        [
            "h3_id",
            "timestamp",
            "value",
        ]
    )
    .collect()
)

print("=" * 80)
print("RAINFALL HISTORY COMPARISON")
print("=" * 80)

for h3_id in sample_ids:

    print(f"H3: {h3_id}")

    history = (
        observations
        .filter(pl.col("h3_id") == h3_id)
        .select(
            [
                "timestamp",
                "value",
            ]
        )
    )

    print(history.head(15))
    print()

RAINFALL HISTORY COMPARISON
H3: 8a7a6325b20ffff
shape: (15, 2)
┌───────────┬────────────┐
│ timestamp ┆ value      │
│ ---       ┆ ---        │
│ str       ┆ f32        │
╞═══════════╪════════════╡
│ 2002-01   ┆ 42.053543  │
│ 2002-02   ┆ 19.736822  │
│ 2002-03   ┆ 95.136787  │
│ 2002-04   ┆ 162.741348 │
│ 2002-05   ┆ 114.5513   │
│ …         ┆ …          │
│ 2002-11   ┆ 153.058014 │
│ 2002-12   ┆ 239.34967  │
│ 2003-01   ┆ 24.273186  │
│ 2003-02   ┆ 15.107536  │
│ 2003-03   ┆ 50.475861  │
└───────────┴────────────┘

H3: 8a7a6325b22ffff
shape: (15, 2)
┌───────────┬────────────┐
│ timestamp ┆ value      │
│ ---       ┆ ---        │
│ str       ┆ f32        │
╞═══════════╪════════════╡
│ 2002-01   ┆ 42.053543  │
│ 2002-02   ┆ 19.736822  │
│ 2002-03   ┆ 95.136787  │
│ 2002-04   ┆ 162.741348 │
│ 2002-05   ┆ 114.5513   │
│ …         ┆ …          │
│ 2002-11   ┆ 153.058014 │
│ 2002-12   ┆ 239.34967  │
│ 2003-01   ┆ 24.273186  │
│ 2003-02   ┆ 15.107536  │
│ 2003-03   ┆ 50.475861  │
└─────────

In [8]:
# ============================================================
# COMPARE DISTANT H3 HISTORIES
# ============================================================

sample_ids = (
    spi["SPI-3"]
    .select("h3_id")
    .unique()
    .sort("h3_id")
    .to_series()
)

indices = [
    0,
    10000,
    20000,
    30000,
    len(sample_ids) - 1,
]

selected = [
    sample_ids[i]
    for i in indices
]

print("=" * 80)
print("DISTANT H3 COMPARISON")
print("=" * 80)

for h3_id in selected:

    print(f"H3: {h3_id}")

    rainfall = (
        observations
        .filter(pl.col("h3_id") == h3_id)
        .select(
            [
                "timestamp",
                "value",
            ]
        )
    )

    print(rainfall.head(15))
    print()

DISTANT H3 COMPARISON
H3: 8a7a6325b20ffff
shape: (15, 2)
┌───────────┬────────────┐
│ timestamp ┆ value      │
│ ---       ┆ ---        │
│ str       ┆ f32        │
╞═══════════╪════════════╡
│ 2002-01   ┆ 42.053543  │
│ 2002-02   ┆ 19.736822  │
│ 2002-03   ┆ 95.136787  │
│ 2002-04   ┆ 162.741348 │
│ 2002-05   ┆ 114.5513   │
│ …         ┆ …          │
│ 2002-11   ┆ 153.058014 │
│ 2002-12   ┆ 239.34967  │
│ 2003-01   ┆ 24.273186  │
│ 2003-02   ┆ 15.107536  │
│ 2003-03   ┆ 50.475861  │
└───────────┴────────────┘

H3: 8a7a6e42cc2ffff
shape: (15, 2)
┌───────────┬────────────┐
│ timestamp ┆ value      │
│ ---       ┆ ---        │
│ str       ┆ f32        │
╞═══════════╪════════════╡
│ 2002-01   ┆ 46.503475  │
│ 2002-02   ┆ 22.287416  │
│ 2002-03   ┆ 89.864883  │
│ 2002-04   ┆ 139.73848  │
│ 2002-05   ┆ 155.792313 │
│ …         ┆ …          │
│ 2002-11   ┆ 116.297577 │
│ 2002-12   ┆ 270.863525 │
│ 2003-01   ┆ 29.383673  │
│ 2003-02   ┆ 17.664042  │
│ 2003-03   ┆ 41.515869  │
└───────────┴───

In [9]:
# ============================================================
# RAINFALL VS SPI (ONE H3)
# ============================================================

h3_id = selected[0]

rainfall = (
    observations
    .filter(pl.col("h3_id") == h3_id)
)

comparison = (
    rainfall
    .join(
        spi["SPI-3"],
        on=["h3_id", "timestamp"],
    )
    .join(
        spi["SPI-6"],
        on=["h3_id", "timestamp"],
    )
    .join(
        spi["SPI-12"],
        on=["h3_id", "timestamp"],
    )
    .select(
        [
            "timestamp",
            "value",
            "spi3",
            "spi6",
            "spi12",
        ]
    )
)

print("=" * 80)
print(f"H3: {h3_id}")
print("=" * 80)
print(comparison)

H3: 8a7a6325b20ffff
shape: (276, 5)
┌───────────┬────────────┬───────────┬──────────┬──────────┐
│ timestamp ┆ value      ┆ spi3      ┆ spi6     ┆ spi12    │
│ ---       ┆ ---        ┆ ---       ┆ ---      ┆ ---      │
│ str       ┆ f32        ┆ f64       ┆ f64      ┆ f64      │
╞═══════════╪════════════╪═══════════╪══════════╪══════════╡
│ 2002-01   ┆ 42.053543  ┆ NaN       ┆ NaN      ┆ NaN      │
│ 2002-02   ┆ 19.736822  ┆ NaN       ┆ NaN      ┆ NaN      │
│ 2002-03   ┆ 95.136787  ┆ -0.004294 ┆ NaN      ┆ NaN      │
│ 2002-04   ┆ 162.741348 ┆ -0.117566 ┆ NaN      ┆ NaN      │
│ 2002-05   ┆ 114.5513   ┆ 0.205188  ┆ NaN      ┆ NaN      │
│ …         ┆ …          ┆ …         ┆ …        ┆ …        │
│ 2024-08   ┆ 2.219894   ┆ 1.047625  ┆ 2.639388 ┆ 2.194909 │
│ 2024-09   ┆ 17.995125  ┆ 1.793562  ┆ 2.757414 ┆ 2.204168 │
│ 2024-10   ┆ 25.309132  ┆ -0.344952 ┆ 1.769559 ┆ 2.372134 │
│ 2024-11   ┆ 207.626907 ┆ 0.477954  ┆ 0.61125  ┆ 2.445797 │
│ 2024-12   ┆ 84.623672  ┆ 0.225401  ┆ 0.349472 ┆

In [10]:
# ============================================================
# RAINFALL, CLIMATOLOGY, ANOMALY & SPI COMPARISON
# ============================================================

BASELINE_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "baselines"
    / "precipitation_amount.parquet"
)

ANOMALY_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "anomalies"
    / "precipitation_amount.parquet"
)

baseline = (
    pl.scan_parquet(BASELINE_FILE)
    .collect()
)

anomalies = (
    pl.scan_parquet(ANOMALY_FILE)
    .with_columns(
        pl.col("timestamp")
        .str.slice(5, 2)
        .cast(pl.UInt8)
        .alias("month")
    )
    .collect()
)

comparison = (
    anomalies
    .filter(pl.col("h3_id") == h3_id)
    .join(
        baseline,
        on=["h3_id", "month"],
        how="left",
    )
    .join(
        spi["SPI-3"],
        on=["h3_id", "timestamp"],
    )
    .join(
        spi["SPI-6"],
        on=["h3_id", "timestamp"],
    )
    .join(
        spi["SPI-12"],
        on=["h3_id", "timestamp"],
    )
    .select(
        [
            "timestamp",
            "value",
            "mean",
            "std",
            "anomaly",
            "z_score",
            "spi3",
            "spi6",
            "spi12",
        ]
    )
)

print("=" * 100)
print(f"H3: {h3_id}")
print("=" * 100)

print(
    comparison.tail(24)
)

H3: 8a7a6325b20ffff
shape: (24, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ timestamp ┆ value     ┆ mean      ┆ std       ┆ … ┆ z_score   ┆ spi3      ┆ spi6      ┆ spi12    │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ str       ┆ f32       ┆ f32       ┆ f32       ┆   ┆ f32       ┆ f64       ┆ f64       ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2023-01   ┆ 22.860664 ┆ 41.211193 ┆ 34.645584 ┆ … ┆ -0.529664 ┆ -0.805529 ┆ -0.962421 ┆ -1.64701 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 5        │
│ 2023-02   ┆ 5.058647  ┆ 34.273003 ┆ 20.449419 ┆ … ┆ -1.428615 ┆ -1.577706 ┆ -1.181182 ┆ -1.73923 │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 5        │
│ 2023-03   ┆ 108.83929 ┆ 95.590385 ┆ 59.926723 ┆ … ┆ 0.

In [11]:
# ============================================================
# ROLLING ACCUMULATION VS SPI
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Rolling accumulation
# ------------------------------------------------------------

def rolling_sum(values, scale):

    out = np.full(len(values), np.nan)

    kernel = np.ones(scale)

    out[scale - 1:] = np.convolve(
        values,
        kernel,
        mode="valid",
    )

    return out


# ------------------------------------------------------------
# Select one H3
# ------------------------------------------------------------

history = (
    comparison
    .with_columns(
        [
            pl.Series(
                "acc3",
                rolling_sum(
                    comparison["value"].to_numpy(),
                    3,
                ),
            ),
            pl.Series(
                "acc6",
                rolling_sum(
                    comparison["value"].to_numpy(),
                    6,
                ),
            ),
            pl.Series(
                "acc12",
                rolling_sum(
                    comparison["value"].to_numpy(),
                    12,
                ),
            ),
        ]
    )
)

print("=" * 120)
print("ROLLING ACCUMULATION VS SPI")
print("=" * 120)

print(
    history.select(
        [
            "timestamp",
            "value",
            "acc3",
            "acc6",
            "acc12",
            "spi3",
            "spi6",
            "spi12",
        ]
    ).tail(24)
)

ROLLING ACCUMULATION VS SPI
shape: (24, 8)
┌───────────┬────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┐
│ timestamp ┆ value      ┆ acc3       ┆ acc6       ┆ acc12     ┆ spi3      ┆ spi6      ┆ spi12     │
│ ---       ┆ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str       ┆ f32        ┆ f64        ┆ f64        ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞═══════════╪════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2023-01   ┆ 22.860664  ┆ 198.089041 ┆ 224.444811 ┆ 462.52104 ┆ -0.805529 ┆ -0.962421 ┆ -1.647015 │
│           ┆            ┆            ┆            ┆ 2         ┆           ┆           ┆           │
│ 2023-02   ┆ 5.058647   ┆ 57.019865  ┆ 229.503458 ┆ 436.45075 ┆ -1.577706 ┆ -1.181182 ┆ -1.739235 │
│           ┆            ┆            ┆            ┆ 4         ┆           ┆           ┆           │
│ 2023-03   ┆ 108.839294 ┆ 136.758606 ┆ 331.0359

In [3]:
# ============================================================
# VERIFY climate_indices INSTALLATION
# ============================================================

import inspect

import climate_indices

print("=" * 60)
print("climate_indices")
print("=" * 60)

print(f"Version : {getattr(climate_indices, '__version__', 'unknown')}")
print()

try:
    from climate_indices import indices

    print("✓ Imported climate_indices.indices")
    print()

    print("Available public functions:")
    for name in sorted(dir(indices)):
        if not name.startswith("_"):
            print(f"  {name}")

    print()
    print("=" * 60)
    print("SPI SIGNATURE")
    print("=" * 60)

    if hasattr(indices, "spi"):
        print(inspect.signature(indices.spi))
    else:
        print("No function named 'spi' found.")

except Exception as e:
    print(type(e).__name__)
    print(e)

climate_indices
Version : 2.4.0

✓ Imported climate_indices.indices

Available public functions:
  Any
  DataShapeError
  Distribution
  Enum
  InvalidArgumentError
  SCALE_MAX
  SCALE_MIN
  annotations
  cast
  check_large_array_memory
  compute
  eddi
  eto
  get_logger
  np
  pci
  percentage_of_normal
  pet
  spei
  spi
  time
  utils

SPI SIGNATURE
(values: 'np.ndarray', scale: 'int', distribution: 'Distribution', data_start_year: 'int', calibration_year_initial: 'int', calibration_year_final: 'int', periodicity: 'compute.Periodicity', fitting_params: 'dict[str, Any] | None' = None) -> 'np.ndarray'


In [6]:
# ============================================================
# COMPARE WITH climate_indices
# ============================================================

from pathlib import Path

import numpy as np
import polars as pl

from climate_indices import compute
from climate_indices import indices
from climate_indices.indices import Distribution

PROJECT_ROOT = Path.cwd().parent.resolve()

OBS_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "observations"
    / "master_observations.parquet"
)

SPI3_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "spi"
    / "spi3.parquet"
)

# ------------------------------------------------------------
# Load observations
# ------------------------------------------------------------

observations = (
    pl.scan_parquet(OBS_FILE)
    .filter(pl.col("variable") == "precipitation_amount")
    .select(
        [
            "h3_id",
            "timestamp",
            "value",
        ]
    )
    .sort(
        [
            "h3_id",
            "timestamp",
        ]
    )
    .collect()
)

spi3 = (
    pl.scan_parquet(SPI3_FILE)
    .sort(
        [
            "h3_id",
            "timestamp",
        ]
    )
    .collect()
)

# ------------------------------------------------------------
# Pick first H3
# ------------------------------------------------------------

h3_id = spi3["h3_id"][0]

rainfall = (
    observations
    .filter(pl.col("h3_id") == h3_id)
)

our_spi = (
    spi3
    .filter(pl.col("h3_id") == h3_id)
)

rain = rainfall["value"].cast(pl.Float64).to_numpy()

our = (
    our_spi["spi3"]
    .cast(pl.Float64)
    .to_numpy()
)

# ------------------------------------------------------------
# Reference SPI
# ------------------------------------------------------------

reference = indices.spi(
    values=rain,
    scale=3,
    distribution=Distribution.gamma,
    data_start_year=2002,
    calibration_year_initial=2002,
    calibration_year_final=2024,
    periodicity=compute.Periodicity.monthly,
)

# ------------------------------------------------------------
# Compare
# ------------------------------------------------------------

mask = np.isfinite(our) & np.isfinite(reference)

diff = our[mask] - reference[mask]

print("=" * 60)
print("SPI-3 REFERENCE COMPARISON")
print("=" * 60)
print(f"H3              : {h3_id}")
print(f"Compared values : {mask.sum()}")
print(f"Mean abs diff   : {np.mean(np.abs(diff)):.10f}")
print(f"Max abs diff    : {np.max(np.abs(diff)):.10f}")
print(f"RMSE            : {np.sqrt(np.mean(diff**2)):.10f}")
print("=" * 60)

comparison = pl.DataFrame(
    {
        "timestamp": rainfall["timestamp"],
        "ours": our,
        "reference": reference,
        "difference": our - reference,
    }
)

print(comparison.tail(20))

2026-07-02T13:20:54.099655Z [info     ] calculation_started            [climate_indices.indices] distribution=gamma index_type=spi input_elements=276 input_shape=(276,) scale=3
2026-07-02T13:20:54.124035Z [info     ] distribution_transform_started [climate_indices.compute] distribution=gamma input_shape='(23, 12)' operation=transform_fitted_gamma periodicity=monthly
2026-07-02T13:20:54.151332Z [info     ] distribution_fitting_started   [climate_indices.compute] calibration_period=2002-2024 distribution=gamma operation=gamma_parameters periodicity=monthly
c:\Projects\ews\.venv312\Lib\site-packages\climate_indices\compute.py:1277: ShortCalibrationWarning: Calibration period is 23 years, which is shorter than the recommended minimum of 30 years. Shorter periods may not capture the full range of climate variability.
  alphas, betas = gamma_parameters(
2026-07-02T13:20:54.234832Z [info     ] distribution_fitting_completed [climate_indices.compute] calibration_period=2002-2024 distribution=g

SPI-3 REFERENCE COMPARISON
H3              : 8a7a6325b20ffff
Compared values : 274
Mean abs diff   : 0.0027390458
Max abs diff    : 0.7337179660
RMSE            : 0.0443256164
shape: (20, 4)
┌───────────┬───────────┬───────────┬────────────┐
│ timestamp ┆ ours      ┆ reference ┆ difference │
│ ---       ┆ ---       ┆ ---       ┆ ---        │
│ str       ┆ f64       ┆ f64       ┆ f64        │
╞═══════════╪═══════════╪═══════════╪════════════╡
│ 2023-05   ┆ -0.349771 ┆ -0.349792 ┆ 0.000021   │
│ 2023-06   ┆ -0.573894 ┆ -0.573956 ┆ 0.000061   │
│ 2023-07   ┆ -1.01744  ┆ -1.017577 ┆ 0.000137   │
│ 2023-08   ┆ -0.473671 ┆ -0.473842 ┆ 0.000172   │
│ 2023-09   ┆ -0.517504 ┆ -0.5176   ┆ 0.000096   │
│ …         ┆ …         ┆ …         ┆ …          │
│ 2024-08   ┆ 1.047625  ┆ 1.047739  ┆ -0.000114  │
│ 2024-09   ┆ 1.793562  ┆ 1.793721  ┆ -0.000158  │
│ 2024-10   ┆ -0.344952 ┆ -0.345003 ┆ 0.00005    │
│ 2024-11   ┆ 0.477954  ┆ 0.477958  ┆ -0.000003  │
│ 2024-12   ┆ 0.225401  ┆ 0.2254    ┆ 0.0000

In [7]:
# ============================================================
# LARGEST DIFFERENCES
# ============================================================

comparison = comparison.with_columns(
    pl.col("difference")
    .abs()
    .alias("abs_difference")
)

print("=" * 60)
print("LARGEST DIFFERENCES")
print("=" * 60)

print(
    comparison
    .sort("abs_difference", descending=True)
    .head(20)
)

LARGEST DIFFERENCES
shape: (20, 5)
┌───────────┬───────────┬───────────┬────────────┬────────────────┐
│ timestamp ┆ ours      ┆ reference ┆ difference ┆ abs_difference │
│ ---       ┆ ---       ┆ ---       ┆ ---        ┆ ---            │
│ str       ┆ f64       ┆ f64       ┆ f64        ┆ f64            │
╞═══════════╪═══════════╪═══════════╪════════════╪════════════════╡
│ 2002-01   ┆ NaN       ┆ NaN       ┆ NaN        ┆ NaN            │
│ 2002-02   ┆ NaN       ┆ NaN       ┆ NaN        ┆ NaN            │
│ 2019-10   ┆ 3.823718  ┆ 3.09      ┆ 0.733718   ┆ 0.733718       │
│ 2018-08   ┆ 2.335501  ┆ 2.335946  ┆ -0.000445  ┆ 0.000445       │
│ 2019-08   ┆ 1.986086  ┆ 1.986442  ┆ -0.000355  ┆ 0.000355       │
│ …         ┆ …         ┆ …         ┆ …          ┆ …              │
│ 2022-07   ┆ -1.482422 ┆ -1.482606 ┆ 0.000184   ┆ 0.000184       │
│ 2024-07   ┆ 2.121153  ┆ 2.121336  ┆ -0.000183  ┆ 0.000183       │
│ 2013-07   ┆ -1.422974 ┆ -1.423151 ┆ 0.000178   ┆ 0.000178       │
│ 2009-08   ┆

In [8]:
# ============================================================
# INVESTIGATE LARGEST DIFFERENCE
# ============================================================

timestamp = "2019-10"

window = (
    comparison
    .filter(
        pl.col("timestamp") >= "2019-05"
    )
    .filter(
        pl.col("timestamp") <= "2020-03"
    )
)

print(window)

shape: (11, 5)
┌───────────┬───────────┬───────────┬────────────┬────────────────┐
│ timestamp ┆ ours      ┆ reference ┆ difference ┆ abs_difference │
│ ---       ┆ ---       ┆ ---       ┆ ---        ┆ ---            │
│ str       ┆ f64       ┆ f64       ┆ f64        ┆ f64            │
╞═══════════╪═══════════╪═══════════╪════════════╪════════════════╡
│ 2019-05   ┆ -1.228645 ┆ -1.228696 ┆ 0.000051   ┆ 0.000051       │
│ 2019-06   ┆ -0.454572 ┆ -0.454626 ┆ 0.000053   ┆ 0.000053       │
│ 2019-07   ┆ -0.2438   ┆ -0.243858 ┆ 0.000058   ┆ 0.000058       │
│ 2019-08   ┆ 1.986086  ┆ 1.986442  ┆ -0.000355  ┆ 0.000355       │
│ 2019-09   ┆ 1.717552  ┆ 1.717702  ┆ -0.00015   ┆ 0.00015        │
│ …         ┆ …         ┆ …         ┆ …          ┆ …              │
│ 2019-11   ┆ 2.644101  ┆ 2.644136  ┆ -0.000034  ┆ 0.000034       │
│ 2019-12   ┆ 2.725864  ┆ 2.725935  ┆ -0.000071  ┆ 0.000071       │
│ 2020-01   ┆ 2.400492  ┆ 2.400571  ┆ -0.000079  ┆ 0.000079       │
│ 2020-02   ┆ 2.472483  ┆ 2.47281

In [9]:
# ============================================================
# DIAGNOSE OCTOBER 2019 SPI-3
# ============================================================

import numpy as np
from scipy.stats import gamma, norm

# ------------------------------------------------------------
# Rolling accumulation
# ------------------------------------------------------------

def rolling_sum(values, scale):

    out = np.full(len(values), np.nan)

    out[scale - 1:] = np.convolve(
        values,
        np.ones(scale),
        mode="valid",
    )

    return out

# ------------------------------------------------------------
# Build monthly table
# ------------------------------------------------------------

rain = rainfall["value"].cast(pl.Float64).to_numpy()
timestamps = rainfall["timestamp"].to_list()

acc3 = rolling_sum(rain, 3)

october = []

for ts, value in zip(timestamps, acc3):

    if ts.endswith("-10") and np.isfinite(value):
        october.append((ts, value))

october = sorted(october)

print("=" * 80)
print("OCTOBER 3-MONTH ACCUMULATIONS")
print("=" * 80)

for ts, value in october:
    print(f"{ts}   {value:.6f}")

# ------------------------------------------------------------
# October 2019 diagnostics
# ------------------------------------------------------------

oct_values = np.array(
    [v for _, v in october],
    dtype=float,
)

positive = oct_values[oct_values > 0]

shape, loc, scale = gamma.fit(
    positive,
    floc=0,
)

target = dict(october)["2019-10"]

q = np.mean(oct_values == 0.0)

g = gamma.cdf(
    target,
    a=shape,
    loc=0,
    scale=scale,
)

h = q + (1 - q) * g

h = np.clip(
    h,
    1e-10,
    1 - 1e-10,
)

spi = norm.ppf(h)

print()
print("=" * 80)
print("2019-10 DIAGNOSTICS")
print("=" * 80)
print(f"Accumulation : {target:.6f}")
print(f"Shape        : {shape:.10f}")
print(f"Scale        : {scale:.10f}")
print(f"Zero prob q  : {q:.10f}")
print(f"Gamma CDF    : {g:.10f}")
print(f"H(x)         : {h:.10f}")
print(f"SPI          : {spi:.10f}")
print("=" * 80)

OCTOBER 3-MONTH ACCUMULATIONS
2002-10   49.505289
2003-10   65.425790
2004-10   58.242712
2005-10   30.829579
2006-10   36.762093
2007-10   45.264037
2008-10   61.810080
2009-10   53.014267
2010-10   47.956856
2011-10   97.311756
2012-10   58.667043
2013-10   45.307247
2014-10   64.244822
2015-10   53.383003
2016-10   32.206301
2017-10   58.965103
2018-10   52.273018
2019-10   220.370831
2020-10   45.713055
2021-10   33.786124
2022-10   26.355770
2023-10   51.766491
2024-10   45.524150

2019-10 DIAGNOSTICS
Accumulation : 220.370831
Shape        : 4.6514402183
Scale        : 12.4756630144
Zero prob q  : 0.0000000000
Gamma CDF    : 0.9999342729
H(x)         : 0.9999342729
SPI          : 3.8237179660


In [10]:
# ============================================================
# INSPECT climate_indices SPI IMPLEMENTATION
# ============================================================

import inspect

from climate_indices import compute
from climate_indices import indices

print("=" * 80)
print("indices.spi")
print("=" * 80)
print(inspect.getsource(indices.spi))

print()

print("=" * 80)
print("compute.transform_fitted_gamma")
print("=" * 80)
print(inspect.getsource(compute.transform_fitted_gamma))

indices.spi
def spi(
    values: np.ndarray,
    scale: int,
    distribution: Distribution,
    data_start_year: int,
    calibration_year_initial: int,
    calibration_year_final: int,
    periodicity: compute.Periodicity,
    fitting_params: dict[str, Any] | None = None,
) -> np.ndarray:
    """
    Computes SPI (Standardized Precipitation Index).

    :param values: 1-D numpy array of precipitation values, in any units,
        first value assumed to correspond to January of the initial year if
        the periodicity is monthly, or January 1st of the initial year if daily
    :param scale: number of time steps over which the values should be scaled
        before the index is computed
    :param distribution: distribution type to be used for the internal
        fitting/transform computation
    :param data_start_year: the initial year of the input precipitation dataset
    :param calibration_year_initial: initial year of the calibration period
    :param calibration_year_final: f